# The TLS226_PERSON_ORIG table 

Welcome to the `TLS226_PERSON_ORIG` table. This table is designed to help users explore and analyse **person-related data** as it appears in patent documents within the PATSTAT database. It is especially useful for users who want to understand *how person information is originally reported* before any data cleaning or standardisation is applied. The `TLS226_PERSON_ORIG` table stores raw information about people and organizations involved in patent applications. Each record represents one person associated with one patent application. A person in this table can be:
- An individual (also called a *natural person*)
- An organisation (also called a *legal person*)

Each row in `TLS226_PERSON_ORIG` provides the details **exactly as reported by the patent office**, including:
- Full name of the person or organisation  
- All available address components, such as:
  - Street
  - City
  - Postal code
  - Country

Because the information is not modified, you may see different spellings, formats, or address structures for what the same person across different patent filings might be.

## Relationship with TLS206_PERSON

Each record in `TLS226_PERSON_ORIG` corresponds directly to a record in the `TLS206_PERSON` table. The key difference between the two tables is the level of processing:
- `TLS226_PERSON_ORIG` 
  - Contains raw, unprocessed data
  - Shows person information exactly as submitted
- `TLS206_PERSON` 
  - Contains cleaned, standardised, and harmonised data
  - Is easier to use for large-scale statistical analysis

## Original (Unprocessed) Nature of the Data
A defining feature of the `TLS226_PERSON_ORIG` table is that the data are kept in their **original form**. This means:
- Names and addresses are not cleaned or standardised
- No harmonisation, normalisation, or disambiguation is applied
- All spelling variations, abbreviations, and formatting differences are preserved

In [7]:
from epo.tipdata.patstat import PatstatClient
from sqlalchemy import func, case, select, and_ 

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS226_PERSON_ORIG, TLS206_PERSON

## Key fields in the ``TLS226_PERSON_ORIG`` table
### PERSON_ID (primary Key)

In the `TLS226_PERSON_ORIG` table, the field `PERSON_ID` plays a central role in connecting raw person data to its standardised counterpart. This identifier can be used to join `TLS226_PERSON_ORIG` with the `TLS206_PERSON` table, allowing users to access the cleaned and harmonised information for individuals and organisations recorded in PATSTAT. This mechanism is similar to what is observed in the `TLS207_PERS_APPLN` notebook, where person identifiers are used to link individuals to patent applications.

The ``PERSON_ID`` serves as the primary key in the ``TLS206_PERSON`` table, uniquely identifying each individual or organisation involved in patent applications. This unique identifier ensures consistency across different patent records, allowing accurate linking of personal data with corresponding roles in patent filings.

### PERSON_ORIG_ID
This field contains a key that uniquely identifies the key to the original (unmodified) record referring to an individual’s data as reported by the patent office. Each unique `PERSON_ORIG_ID` is linked to a corresponding `PERSON_ID`. The same `PERSON_ID` may appear multiple times, because a person who is recorded in slightly different ways by national patent offices (for example, due to spelling variations or address differences) can later be standardised, harmonised, and consolidated into a single unified entity.

In [14]:
q = (
    db.query(
        TLS226_PERSON_ORIG.person_orig_id,
        TLS226_PERSON_ORIG.person_id,
    )
    .order_by(
        TLS226_PERSON_ORIG.person_orig_id
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,person_orig_id,person_id
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
...,...,...
995,996,987
996,997,988
997,998,989
998,999,990


## Name fields
The following sections explain how names are stored within the dataset. Five distinct fields capture specific details, ranging from structured name parts to original language scripts.
### NAME_FREEFORM
This field contains the full name represented as a single text string. Instead of being separated into distinct columns (like first, middle, and last), the entire name is stored here.

### PERSON_NAME_ORIG_LG
This field stores the person's name in the original language (non-transliterated). This is used for names requiring specific character sets outside the standard Latin alphabet. Examples include Japanese, Chinese, Korean, Arabic, or Cyrillic characters.

### LAST_NAME
This field contains the last name (surname/family name) of a natural person **or** the name of a legal person (organization).

### FIRST_NAME
This field contains the first name (given name) of a natural person. This data point applies exclusively to natural persons.

### MIDDLE_NAME
This field contains the middle name of a natural person. This data point applies exclusively to natural persons.

In [16]:
q = (
    db.query(
        TLS226_PERSON_ORIG.name_freeform,
        TLS226_PERSON_ORIG.person_name_orig_lg,
        TLS226_PERSON_ORIG.last_name,
        TLS226_PERSON_ORIG.first_name,
        TLS226_PERSON_ORIG.middle_name
    )
    .distinct()
    .limit(10000)
)

res = patstat.df(q)
res

,name_freeform,person_name_orig_lg,last_name,first_name,middle_name
0,"BROWN, CLIFFORD, E., JR.",,,,
1,Kharitonova T.V.,Харитонова Т.В.,,,
2,"NELSON, Aaron, Basil",,,,
3,,,Simeoni,Denis,
4,,,Jang,Jungshik,
...,...,...,...,...,...
9995,徐远君,徐远君,,,
9996,"Sakai, Kenji",,,,
9997,Осипов Владимир Гаврилович,,,,
9998,PLOSHKIN A.V.,,,,


## Address fields
The following sections detail how location data is structured. These fields allow for the storage of addresses in both unstructured formats (single text strings) and structured formats (specific data points for city, street, etc.), as well as specific country codes for legal and postal purposes.

### ADDRESS_FREEFORM
This field contains the complete address stored as a single text string. This format is utilised when the address data is not available in a structured form. If the individual components—such as street, city, or zip code—are not separated into distinct fields, the entire address is recorded here.

### ADDRESS_1 through ADDRESS_5
These five fields (`ADDRESS_1`, `ADDRESS_2`, `ADDRESS_3`, `ADDRESS_4`, `ADDRESS_5`) represent specific lines of an address in sequential order. These fields contain the first through fifth lines of a person's address. They are used to capture multi-line address details exactly as they would appear on an envelope or label. Fields after ´ADDRESS_3´ are usually left blank.

### STREET
This field contains specifically the street portion of the address. This is a structured field used to isolate the street name and number from the rest of the address data.

### CITY
This field contains the city portion of the address. This is a structured field used to isolate the city or municipality name.

### ZIP_CODE
This field contains the zip code (also known as postal code or postcode). It stores the alphanumeric or numeric code used for mail sorting and delivery purposes.

### STATE
This field contains the US state associated with the address. This field is specifically designated for recording the state component of addresses located within the United States.

In this example, we start from records with ´PERSON_ID´ > 150&nbsc;000 to avoid earlier entries, which often contain many null or missing fields.

In [19]:
q = (
    db.query(
        TLS226_PERSON_ORIG.address_freeform,
        TLS226_PERSON_ORIG.address_1,
        TLS226_PERSON_ORIG.address_2,
        TLS226_PERSON_ORIG.address_3,
        TLS226_PERSON_ORIG.address_4,
        TLS226_PERSON_ORIG.address_5,
        TLS226_PERSON_ORIG.street,
        TLS226_PERSON_ORIG.city,
        TLS226_PERSON_ORIG.zip_code,
        TLS226_PERSON_ORIG.state,
    )
    .where(
        TLS226_PERSON_ORIG.person_id > 1500000
    )
    .distinct()
    .limit(10000)
)

res = patstat.df(q)
res

,address_freeform,address_1,address_2,address_3,address_4,address_5,street,city,zip_code,state
0,,Rain 26,CH-3373 Heimenhausen,,,,,,,
1,,,,,,,,Rockville,,MD
2,,,,,,,,Maple Grove,,MI
3,,"3700 Regency Parkway, Suite 120","Cary, NC 27518",,,,,,,
4,,,,,,,,Cookeville,,TN
...,...,...,...,...,...,...,...,...,...,...
9995,,Nijverheidsweg 4,3534 AM Utrecht,,,,,,,
9996,,Graf-Beust-Allee 17,45141 Essen,,,,,,,
9997,,,,,,,,Wyckoff,,
9998,,"Barncroft, Vicarage Lane Dore",Sheffield S17 3GX,,,,,,,


The remaining fields specify the geographic associations of the person or entity.

### PERSON_CTRY_CODE
This field identifies the country part of the correspondence address for the person or business. The standard format of the ´PERSON_CTRY_CODE´ is as follows:
*   Domain: 2 uppercase characters (A-Z).
*   Standard: Strictly follows the WIPO ST.3 standard (e.g., "GB" for the United Kingdom).
*   Exceptions: In cases of data quality issues, non-compliant characters may appear (e.g., "UK" instead of the standard "GB").

### RESIDENCE_CTRY_CODE
This field identifies the country of the person's actual residence.

In [21]:
q = (
    db.query(
        TLS226_PERSON_ORIG.person_ctry_code,
        TLS226_PERSON_ORIG.residence_ctry_code
    )
    .where(
        TLS226_PERSON_ORIG.person_id > 1500000
    )
    .distinct()
    .limit(10000)
)

res = patstat.df(q)
res

,person_ctry_code,residence_ctry_code
0,UA,
1,BM,
2,FI,FI
3,BB,
4,MC,
...,...,...
1059,ZO,
1060,IL,DE
1061,CA,AR
1062,DE,ES


## Extra fields

### SOURCE
This column identifies the source that provided the data. It is always a short, 5-letter code.

*   DOCDB: European Patent Office (Bibliographic Database)
*   EPREG: European Patent Register
*   USPTO: United States Patent & Trademark Office

### SOURCE_VERSION
This column provides extra detail, but **only** for data provided by the USPTO. It provides information on which software version or file batch was used.

In [11]:
q = (
    db.query(
        TLS226_PERSON_ORIG.source,
        TLS226_PERSON_ORIG.source_version,
    )
    .order_by(
        TLS226_PERSON_ORIG.source,
        TLS226_PERSON_ORIG.source_version
    )
    .distinct()
    .limit(10000)
)

res = patstat.df(q)
res

,source,source_version
0,DOCDB,
1,EPREG,
2,USPTO,BACKFILE
3,USPTO,V4.2
4,USPTO,V4.3
5,USPTO,V4.4
6,USPTO,V4.5
7,USPTO,V4.6
8,USPTO,V4.7
9,USPTO,V4.72


### ROLE
This field provides specific classification details regarding the entity receiving the patent rights (the assignee) as determined by the USPTO. It is assigned directly by the USPTO to categorise the nature or legal status of the assignee within the patent record.

In [25]:
q = (
    db.query(
        TLS226_PERSON_ORIG.role,
    )
    .order_by(
        TLS226_PERSON_ORIG.role
    )
    .distinct()
)

res = patstat.df(q)
res

,role
0,
1,
2,0
3,01
4,02
5,03
6,04
7,05
8,06
9,07
